# 23 - Python 3.14 New Features


## 1. Template String Literals (`t-strings`) (PEP 750)

Python 3.14 ile tanıtılan template string sabitleri (`t'...'`), f-string sözdizimini kullanmasına rağmen ifadeyi anında `str` olarak birleştirmek yerine statik metin parçalarını ve interpolasyon nesnelerini (`string.templatelib.Template` ve `Interpolation`) ayrı ayrı tutar.

**Syntax**

```Python
from string.templatelib import Template, Interpolation

user_input = '<script>alert(1)</script>'
tmpl: Template = t'<div>{user_input}</div>'
```

**Neden `t-strings` Kullanılır?**

* **Enjeksiyon Saldırılarına Karşı Güvenlik:** SQL sorguları, HTML şablonları veya kabuk (shell) komutları oluşturulurken kullanıcı girdileri doğrudan metne yapıştırılmaz; işleyici fonksiyon `Template` içindeki `Interpolation` öğelerini dolaşarak her değeri bağlama göre otomatik kaçışlar (escape) veya parametreleştirir.
* **Zengin İfade Erişimi:** Her `Interpolation` nesnesi; değerlendirilen değeri (`.value`), kaynak ifade metnini (`.expression`), dönüşüm bayrağını (`.conversion`: `'s'`, `'r'`, `'a'`) ve biçim belirticisini (`.format_spec`) korur.


In [ ]:
import html
from string.templatelib import Interpolation, Template


def render_safe_html(template: Template) -> str:
    parts: list[str] = []
    for item in template:
        if isinstance(item, Interpolation):
            parts.append(html.escape(str(item.value)))
        else:
            parts.append(item)
    return ''.join(parts)


untrusted_name: str = '<img src=x onerror=alert(1)>'
page_template: Template = t'<p>Hoş geldin, {untrusted_name}!</p>'

print('Static strings:', page_template.strings)
print('Interpolations:', [i.expression for i in page_template.interpolations])
print('Sanitized HTML:', render_safe_html(page_template))


## 2. Deferred Evaluation of Annotations (PEP 649 & PEP 749)

Python 3.14 ile fonksiyon, sınıf ve modül düzeyindeki tip ek açıklamaları (annotations) artık tanım anında hemen değerlendirilmez; yalnızca `__annotations__` veya `annotationlib.get_annotations()` üzerinden ihtiyaç duyulduğunda çalıştırılan özel bir `__annotate__` fonksiyonu içinde ertelenmiş (lazy) olarak saklanır.

**Syntax**

```Python
from annotationlib import Format, get_annotations


class Node:
    def connect(self, target: Node) -> Node:
        return target
```

**Kaldırılan Kısıtlamalar ve `annotationlib` Araçları**

* **İleri Referanslarda (Forward References) Tırnak Zorunluluğunun Kalkması:** Sınıf gövdesi içinde kendi sınıf adını (`target: Node`) veya henüz tanımlanmamış tipleri tırnak içine almadan (`'Node'`) ve `from __future__ import annotations` kullanmadan doğrudan yazmak geçerlidir.
* **Esnek İnceleme Formatları (`Format`):** `annotationlib.get_annotations(obj, format=...)` ile açıklamalar gerçek nesne olarak (`Format.VALUE`), ileri referansları metin tutup diğerlerini çözerek (`Format.FORWARDREF`) veya ham kaynak metin olarak (`Format.STRING`) okunabilir.


In [ ]:
from annotationlib import Format, get_annotations


class LinkedNode:
    def __init__(self, value: int, next_node: LinkedNode | None = None) -> None:
        self.value = value
        self.next_node = next_node

    def attach(self, node: FutureNode) -> LinkedNode:
        return self


class FutureNode:
    pass


print('VALUE format:', get_annotations(LinkedNode.__init__, format=Format.VALUE))
print('STRING format:', get_annotations(LinkedNode.attach, format=Format.STRING))


## 3. Multiple Interpreters in the Standard Library (`concurrent.interpreters`) (PEP 734)

Python 3.14 ile CPython'ın alt yorumlayıcı (subinterpreter) altyapısı standart kütüphanedeki `concurrent.interpreters` modülü ve `InterpreterPoolExecutor` aracılığıyla doğrudan Python koduna açılmış; her biri kendi izole GIL kilidine ve belleğine sahip yorumlayıcılarla gerçek çok çekirdekli paralellik sağlanmıştır.

**Syntax**

```Python
from concurrent.futures import InterpreterPoolExecutor
from concurrent import interpreters

interp = interpreters.create()
interp.exec('x = 10 + 20')
```

**Mimari Avantajlar**

* **İzole Bellek ve Bağımsız GIL:** Standart `ThreadPoolExecutor` tek bir GIL paylaşırken ve `ProcessPoolExecutor` ağır işletim sistemi süreçleri oluştururken, `InterpreterPoolExecutor` aynı süreç içinde hafif ve bellek açısından izole yorumlayıcı havuzu çalıştırır.
* **Paylaşılabilir Veri Aktarımı:** Yorumlayıcılar arasında ilkel tipler (`int`, `str`, `bytes`, `tuple`) ve `memoryview` verileri güvenli kanallar (queues) veya görev argümanları üzerinden aktarılır.


In [ ]:
from concurrent import interpreters
from concurrent.futures import InterpreterPoolExecutor

interp = interpreters.create()
interp.exec('result = sum(i * i for i in range(100))')
interp.close()


def compute_chunk(n: int) -> int:
    return sum(i * i for i in range(n))


with InterpreterPoolExecutor(max_workers=4) as executor:
    totals: list[int] = list(executor.map(compute_chunk, [50_000, 60_000, 70_000, 80_000]))

print('InterpreterPoolExecutor results:', totals)


## 4. Bracketless Exception Tuple Syntax (PEP 758) & `finally` Control-Flow Restriction (PEP 765)

Python 3.14 ile birden fazla istisna türünü yakalarken parantez kullanma zorunluluğu kaldırılmış (`except ValueError, KeyError:`), ayrıca `finally` blokları içinden `return`, `break` ve `continue` ile sessizce istisna yutmayı önleyen sözdizimi uyarısı getirilmiştir.

**Syntax**

```python
try:
    parse_record(raw)
except ValueError, KeyError:
    handle_invalid_record()
```

**Sözdizimi Kuralları**

* **Parantezsiz `except` (PEP 758):** `except TypeError, ValueError:` doğrudan geçerlidir; ancak yakalanan istisnayı bir değişkene bağlarken (`as err`) belirsizliği önlemek için parantez (`except (TypeError, ValueError) as err:`) zorunluluğu devam eder.
* **`finally` Bloğunda Akış Kısıtlaması (PEP 765):** `finally:` bloğu içinden `return`, `break` veya `continue` ile çıkmak aktif istisnayı sessizce bastırdığı için artık `SyntaxWarning` üretir.


In [ ]:
def extract_port(config: dict[str, str]) -> int:
    try:
        return int(config['port'])
    except KeyError, ValueError:
        return 8080


print('Missing key fallback:', extract_port({}))
print('Invalid int fallback:', extract_port({'port': 'abc'}))
print('Valid port:', extract_port({'port': '9090'}))


## 5. Zstandard Compression (`compression.zstd`) (PEP 784), Zero-Overhead External Debugger (PEP 768) & Tail-Call Interpreter

Python 3.14 standart kütüphanesine Meta'nın yüksek hızlı Zstandard sıkıştırma algoritması (`compression.zstd`) dahil edilmiş, çalışan süreçlere sıfır performans kaybıyla bağlanabilen harici hata ayıklayıcı arayüzü (`sys.remote_exec`) eklenmiş ve CPython yorumlayıcı döngüsü kuyruk çağrısı (tail-call) mimarisiyle hızlandırılmıştır.

**Syntax**

```python
from compression import zstd

compressed = zstd.compress(b'raw payload')
original = zstd.decompress(compressed)
```

**Önemli Eklemeler**

* **`compression` Ad Alanı ve `zstd` (PEP 784):** Yeni `compression` paketi altında `zstd` modülü (`compress`, `decompress`, `open`, `ZstdCompressor`, `ZstdDecompressor`) standart hale gelmiş; mevcut `gzip`, `bz2`, `lzma` ve `zlib` modülleri de `compression.*` altında yeniden dışa aktarılmıştır.
* **Sıfır Maliyetli Harici Hata Ayıklama (PEP 768):** `sys.remote_exec(pid, script_path)` ve `python -m pdb -p <pid>` komutları, üretim ortamında çalışan bir Python sürecini yeniden başlatmadan veya yavaşlatmadan güvenli noktada incelemeyi sağlar.
* **Kuyruk Çağrılı Yorumlayıcı ve Free-Threaded Resmi Desteği:** PEP 779 kapsamında free-threaded Python (No-GIL) deneysel aşamadan çıkarılarak resmi destek kapsamına alınmış; uyumlu derleyicilerde (Clang 19+, GCC 14+) tail-call yorumlayıcı mimarisi tek iş parçacıklı performansı da artırmıştır.


In [ ]:
from compression import zstd
import sys

raw_payload: bytes = b'Python 3.14 Zstandard compression benchmark data. ' * 200
compressed_data: bytes = zstd.compress(raw_payload, level=3)
restored_data: bytes = zstd.decompress(compressed_data)

assert restored_data == raw_payload
print('Python version:', sys.version.split()[0])
print('Original bytes:', len(raw_payload))
print('Zstd compressed bytes:', len(compressed_data))
print('Compression ratio:', f'{len(raw_payload) / len(compressed_data):.2f}x')
